# Cascade evaluation and comparison

**SERS-D2DNet + SuperRaman · v0.3.0**

[Notebook guide](../docs/NOTEBOOK_GUIDE.md) · [Paper alignment](../docs/PAPER_IMPLEMENTATION_MAP.md) · [README](../README.md)

## Goal
Connect completed checkpoints, preserve input identities, and inspect before/after predictions.

## Setup
Public notebooks start without saved execution outputs. Select the Python environment where `.[notebooks]` is installed. Edit `CONFIG` below to select the demo or your research configuration. Every notebook opens the same persistent run. Configuration changes require a new `run_directory`. The default executes a synthetic, one-epoch, width-2 demonstration across all four portable devices.

In [ ]:
CONFIG = "configs/workflow_demo.json"
RUN_RESEARCH_TRAINING = False

In [ ]:
import numpy as np
from sersd2d_superraman import __version__
from sersd2d_superraman.provenance import read_json
from sersd2d_superraman.data import load_batch
from sersd2d_superraman.notebook import (
    open_workflow,
    display_table,
    setup_plotting,
    training_plot,
    spectrum_plot,
    confusion_plot,
)

workflow = open_workflow(CONFIG)
PROJECT, settings = workflow.project, workflow.config
MODE = settings["mode"]
RUN_TRAINING = MODE == "demo" or RUN_RESEARCH_TRAINING
setup_plotting()
display_table(
    ["Control", "Value"],
    [
        ["Software", __version__],
        ["Mode", MODE],
        ["Devices", ", ".join(settings["sources"])],
        [
            "Translation width / epochs",
            f"{settings['translation_config']['model']['width']} / {settings['translation_config']['training']['epochs']}",
        ],
        ["Classifier epochs", settings["classifier_config"]["training"]["epochs"]],
        [
            "Classifier / translation device",
            f"{settings['classifier_device']} / {settings['translation_device']}",
        ],
        ["Research training requested", RUN_RESEARCH_TRAINING],
    ],
)

### Read the execution scope
The demo verifies software behavior. Its plots and scores are synthetic and have no scientific performance interpretation. Research mode uses authorized preprocessed arrays and the recovered width-128 graph. Published-result reproduction remains unresolved; review the paper alignment document before reporting results.

## Steps
### 1. Prepare and audit the input roles
The interface accepts preprocessed `[N,1400]` spectra in `[0,1]`. Rows retain device, analyte, sample identity, optional Raman axis, and optional acquisition group. Class-matched training pairs are physically unpaired. Split checks keep duplicates and acquisition groups together.

In [ ]:
counts = workflow.prepare()
display_table(
    ["Device", "Fit", "Validation", "Test"],
    [[d, r["train"], r["validation"], r["test"]] for d, r in counts.items()],
)

### 4. Connect the models and evaluate the held-out cascade
A bundle copies one classifier and the selected translators. In separate workers, the same test rows follow two paths: original → classifier, and source → translator `out` → per-row `[-1,1]` classifier transform → classifier. Class-mean spectral agreement uses independent Renishaw test spectra.

In [ ]:
ready = (workflow.output / "classification/checkpoint/metadata.json").is_file() and all(
    (workflow.output / "translation" / d / "checkpoint/metadata.json").is_file() for d in settings["sources"]
)
reports = {}
if ready:
    workflow.build_bundle()
    reports = workflow.evaluate()
    display_table(
        ["Device", "Original accuracy", "Translated accuracy", "MAE before", "MAE after"],
        [
            [
                d,
                round(r["classification"]["original"]["accuracy"], 4),
                round(r["classification"]["translated"]["accuracy"], 4),
                round(r["translation"]["before"]["mae"], 5),
                round(r["translation"]["after"]["mae"], 5),
            ]
            for d, r in reports.items()
        ],
    )
else:
    print("Evaluation pending: complete both training stages before building the bundle.")

### 5. Inspect a class and the classification comparison
Change `DEVICE` and `CLASS_ID` to inspect the same held-out cohort. Shading shows between-spectrum standard deviation, not a confidence interval. A physical Raman axis is shown only when supplied; otherwise the axis is bin index.

In [ ]:
DEVICE = settings["sources"][0]
CLASS_ID = 0
if reports:
    spectrum_plot(workflow, DEVICE, CLASS_ID)
    import matplotlib.pyplot as plt

    plt.show()
    confusion_plot(workflow, DEVICE)
    plt.show()

## Checks
Artifact reuse checks package-source and file hashes plus input/configuration identities. Full worker logs are stored under the run directory in logs/. Test rows are evaluated after checkpoint selection; they do not choose the models.

In [ ]:
display_table(["Stage", "Status"], workflow.status().items())

## Next Steps
For real experiments, copy `configs/workflow_research.json` to `configs/local/my_run.json`, set the data path and a new run directory, and select it in `CONFIG`. Set RUN_RESEARCH_TRAINING = True explicitly after reviewing the displayed settings. See the [notebook guide](../docs/NOTEBOOK_GUIDE.md) for stage navigation and interpretation.